### Imports

Loaded required libraries for data manipulation.

In [1]:
# Imports
import pandas as pd

### Display Setings

Configured pandas to display all columns in output.

In [ ]:
# Display settings
pd.set_option('display.max_columns', None)

### Data Loading

Loaded the cleaned loan data from parquet file.

In [2]:
# Data loading
loans = pd.read_parquet('../data/cleaned/parquet/accepted.parquet')

Displayed first few rows of the dataset to inspect its structure.

In [3]:
loans.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies
0,32000.0,60,10.49,687.65,B,B3,Public Service,10,MORTGAGE,120000.0,...,1981-10-01,20.0,0.0,39687.0,57.8,42.0,w,Individual,2.0,0.0
1,9600.0,36,12.99,323.42,C,C1,Teacher,10,RENT,21900.0,...,2001-04-01,13.0,1.0,4509.0,38.9,20.0,w,Individual,0.0,1.0
2,4000.0,36,6.68,122.93,A,A3,System Analyst,4,MORTGAGE,83000.0,...,2003-09-01,16.0,0.0,1564.0,17.2,25.0,w,Individual,2.0,0.0
3,6025.0,36,10.91,197.00,B,B4,Admin assistant,10,RENT,52000.0,...,2005-06-01,11.0,0.0,2706.0,12.8,25.0,w,Individual,0.0,0.0
4,25000.0,60,26.30,752.96,E,E5,Coordinator,10,OWN,65000.0,...,1999-07-01,19.0,0.0,49461.0,24.7,33.0,w,Individual,0.0,0.0


### Feature Engineering

Created credit history duration feature. Following the logic:
- Calculate the difference in days between the issue date and the earliest credit line
- Convert the difference to years
- Round the result to 2 decimal places

In [4]:
# Calculate credit history in years (difference between issue date and earliest credit line)
loans['credit_history'] = ((loans['issue_d'] - loans['earliest_cr_line'])
                                   .dt.total_seconds() / (365.25 * 24 * 60 * 60)).round(2)

Created account velocity metric. Following the logic:
- Divide the total number of accounts by the credit history in years
- Round the result to 2 decimal places

In [5]:
# Calculate credit account velocity (accounts opened per year of history)
loans['acc_to_history_ratio'] = (loans['total_acc'] / loans['credit_history']).round(2)

Created monthly debt burden indicator. Following the logic:
- Divide the installment by the monthly income
- Round the result to 2 decimal places

In [6]:
# Calculate monthly debt burden (installment relative to monthly income)
loans['debt_burden_ratio'] = (loans['installment'] / (loans['annual_inc'] / 12)).round(2)

Created binary default flag. Following the logic:
- Check if the loan status is in the list of default statuses
- Convert the result to an integer
- 1 if default, 0 otherwise

In [7]:
# Create flag variable for default
# Define default statuses
default_statuses = [
    'Charged Off',                                    
    'Default',                                        
    'Late (31-120 days)',                             
    'Late (16-30 days)',                              
    'Does not meet the credit policy. Status:Charged Off'
]

# Create flag for default 
loans['is_default'] = loans['loan_status'].isin(default_statuses).astype(int)

Created high debt-to-income flag. Following the logic:
- Check if the debt-to-income ratio is greater than 30
- Convert the result to an integer
- 1 if high DTI, 0 otherwise

In [8]:
# Create flag for high debt-to-income ratio
loans['is_high_dti'] = (loans['dti'] > 30).astype(int)

Created high revolving utilization flag. Following the logic:
- Check if the revolving utilization is greater than 60
- Convert the result to an integer
- 1 if high revolving utilization, 0 otherwise

In [9]:
# Create flag for high revolving utilization
loans['is_high_util'] = (loans['revol_util'] > 60).astype(int)

Created high-risk grade flag. Following the logic:
- Check if the grade is in the list of high-risk grades
- Convert the result to an integer
- 1 if high-risk grade, 0 otherwise

In [10]:
# Create flag for high-risk grade
loans['is_high_risk_grade'] = loans['grade'].isin(['E', 'F', 'G']).astype(int)

Created unverified income flag. Following the logic:
- Check if the verification status is 'Not Verified'
- Convert the result to an integer
- 1 if unverified income, 0 otherwise

In [11]:
# Create flag for unverified income
loans['is_unverified'] = (loans['verification_status'] == 'Not Verified').astype(int)

Displayed first 5 rows of the transformed dataset to validate the new features.

In [12]:
loans.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,mort_acc,pub_rec_bankruptcies,credit_history,acc_to_history_ratio,debt_burden_ratio,is_default,is_high_dti,is_high_util,is_high_risk_grade,is_unverified
0,32000.0,60,10.49,687.65,B,B3,Public Service,10,MORTGAGE,120000.0,...,2.0,0.0,33.34,1.26,0.07,0,0,0,0,0
1,9600.0,36,12.99,323.42,C,C1,Teacher,10,RENT,21900.0,...,0.0,1.0,13.08,1.53,0.18,0,0,0,0,0
2,4000.0,36,6.68,122.93,A,A3,System Analyst,4,MORTGAGE,83000.0,...,2.0,0.0,11.58,2.16,0.02,0,0,0,0,1
3,6025.0,36,10.91,197.00,B,B4,Admin assistant,10,RENT,52000.0,...,0.0,0.0,12.50,2.00,0.05,0,0,0,0,1
4,25000.0,60,26.30,752.96,E,E5,Coordinator,10,OWN,65000.0,...,0.0,0.0,18.59,1.78,0.14,0,1,0,1,0


### Export

Saved the processed dataset to parquet format.

In [13]:
loans.to_parquet('../data/processed/parquet/accepted.parquet', index=False)